In [1]:

# For newbies
# python3.12 -m venv python
# source dlplnenv/bin/activate
# pip install --upgrade pip
# pip install numpy torchvision seaborn torchviz matplotlib torch jupyter jupyter_contrib_nbextensions

# Guión de prácticas: Introducción breve a Tensores en Pytorch

**Asignatura**: Deep Learning para procesamiento de Lenguaje Natural, 2026/2027

**Profesor**: Juan A. Botía (juanbot@um.es)

**Máster de Inteligencia Artificial**

**Facultad de Informática**

![](https://www.um.es/image/layout_set_logo?img_id=175281&t=1726728636242)

**Universidad de Murcia**

![](https://www.um.es/o/um-lr-principal-um-home-theme/images/logo-um.png)




# Introducción

Pytorch es una librería desarrollada en Python para trabajo especializado en el desarrollo de modelos de deep learning. En la actualidad, hay dos librerías que impulsan la mayoría de las innovaciones producidas en el mundo del PLN (Procesamiento de Lenguaje Natural) basado en deep learning. 

* Por un lado está **tensorflow**, desarrollada por [Google](https://www.tensorflow.org/) y con una inmensa comunidad de desarrolladores sobre la que se ha construido una de las librerías de deep learning de alto nivel más populares si no la más popular, [Keras](https://keras.io/). Aunque ahora también podemos usar Pytorch bajo Keras. 

* Por otro lado, tenemos **Pytorch**, desarrollada por FAIR (Facebook´s AI Research) y que, en la actualidad, es la preferida por la industria y la academia. La razón, en este punto de la asignatura, es algo compleja de explicar pero digamos que permite más flexibilidad a la hora de trabajar con la estructura de las redes neuronales que podamos crear. 

En todo caso, Pytorch nos permite desarrollar modelos de deep learning mediante la definición de arquitecturas de redes neuronales altamente especializadas, mecanismos de entrenamiento específicos para las arquitecturas definidas eficientes para el hardware que estemos usando en cada momento y para optimizar nuestros modelos. 

Podemos correr Pytorch en AWS, GCP y Azure entre otras. Podemos correr Pytorch en notebooks Google Collab y también en Kaggle, lo que quiere decir que podemos correrlo directamente en nuestro navegador sin instalar software adicional alguno. El tipo de hardware soportado incluye  CPU, GPU y TPU. Además, podemos usar funcionalidad para entrenar modelos con múltiples GPUs en varias máquinas. Todo esto y mucho más.

En esta sesión veremos, a partir de un extracto de los tutoriales que podemos encontrar en la página oficial de [Pytorch](<https://pytorch.org/tutorials), una introducción a los tensores, el elemento básico de información del deep learning.

En la tercera sesión de laboratorio veremos cómo se materializan en Pytorch los grafos de computación que veremos también en una sesión de teoría.


# Tensores

Los tensores son estructuras de datos matriciales multidimensionales usadas para representar y almacenar

* los datos de entrada a nuestros modelos,

* los parámetros de nuestros modelos,

* los valores de activación que se van generando como consecuencia de operaciones de inferencia forward,

* los valores del operador gradiente a aplicar a los parámetros de nuestros modelos cuando entrenamos,  

* las salidas de la red, entre otros.

Por similitud en la estructura de datos, **un tensor es un array n-dimensional**. Por tanto, son muy similares al tipo de datos [ndarray de NumPy](https://numpy.org/doc/stable/reference/arrays.ndarray.html). Pero los tensores están preparados, entre otras cosas, para sen eficientemente cargados y procesados en hardware especialiado de Inteligencia Artificial, las GPUs y TPUs (y NPUs). Las APIs de una y otra son muy similares en todo caso. 

Vamos a ver una serie de ejemplos de uso que van a ilustrar las ideas que estamos comentando en clase.
Este tutorial está basado en <https://pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html>.

Necesitamos empezar usando Pytorch y NumPy.
La librería de Pytorch se llama `torch`. Si no la hemos instalado, la instalamos con `pip` y ya la podremos importar para usarla. Además necesitamos `NumPy`. Para que Pytorch esté contento con NumPy, si nos da problemas la versión por defecto, podemos decirle que instale una versión menor, por ejemplo la 1.24, simplemente haciendo

```python
pip install numpy==1.24
```

Lo hacemos como sigue

In [2]:
import torch
import numpy as np

## Inicialización de tensores

Podemos inicializar tensores de múltiples formas:

* desde código indicando explícitamente estructura y valores para cada celda, 

* desde arrays NumPy, 

* desde otros tensores, 

* generando números aleatorios para su inicialización o 

* mediante constantes o secuencias entre otros. 

Vamos a ver ejemplos de todos ellos.

### Directamente desde código

En este caso, indicamos tal cual qué queremos que contenga el tensor. El tipo de datos se deduce, cuando es posible, directamente desde el código.


In [3]:
data = [[1, 2], [3, 4]]
print(data)
print(type(data))
x_data = torch.tensor(data)
print(x_data)

[[1, 2], [3, 4]]
<class 'list'>
tensor([[1, 2],
        [3, 4]])


Como vemos, aparentemente, el único efecto que de momento tiene el usar un tensor es el de integrar la lista de dos elementos, que a su vez son listas de elementos, dentro de una clase envoltorio. Vamos a ver que es mucho más que eso pronto.

### Desde un array NumPy

Podemos intercambiar tensores por arrays NumPy y viceversa. Esto tiene su importancia, sobre todo cuando necesitamos que un array NumPy se integre de alguna forma en las operaciones o parámetros de una red neuronal con Pytorch.
En este ejemplo, primero convertimos nuestra lista `data` de arriba en un array NumPy. Luego desde NumPy, usando una llamada Pytorch, lo integramos en un tensor 


In [4]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
print(x_np)

tensor([[1, 2],
        [3, 4]])


En este ejemplo de abajo, aunque nos estamos adelantando un poco, vamos a ver un ejemplo en el que al intentar usar un array NumPy en un entorno Pytorch, nos produciría un error. En este caso estamos intentando ejecutar una operación típica de NumPy en un tensor. Pero la lógica de negocio del método `forward()` exige que toda operación que se haga para modificar `x` se registre en un grafo computacional para poder optimizar los parámetros de la red de forma automática, algo que `np.exp(x)` no incorpora pues no ha sido diseñada para eso.

In [1]:
import torch
import torch.nn as nn
import numpy as np
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 1)

    def forward(self, x):
        x = self.linear(x)
        # ERROR: trying to use a NumPy operation on a Tensor that requires gradients
        x = torch.exp(x)
        return x
    
model = MyModel()
x = torch.randn(10, 4)
y = model(x)


Entonces, para solucionar este problema de forma sencilla, simplemente introduciríamos la llamada correspondiente que sí incorpora la gestión de gradientes, como aquí

In [ ]:
def forward(self, x):
    x = self.linear(x)
    x = torch.exp(x)
    return x


### Desde otro tensor 

No estamos haciendo una copia del contenido, solo de la estructura. 
En este caso, cuando creamos un tensor a partir de otro, por defecto el nuevo tensor retiene las propiedades (forma y tipo de datos) del tensor que se pasa como parámetro. Pero podemos indicar un tipo básico nuevo para la celda del tensor, como en el segundo ejemplo.


In [ ]:
x_ones = torch.ones_like(x_data) # al no indicar nada, retoma el tipo original
print(f"Tensor de unos: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # genera aleatorios flotantes
print(f"Tensor de aleatorios: \n {x_rand} \n")

Tensor de unos: 
 tensor([[1, 1],
        [1, 1]]) 

Tensor de aleatorios: 
 tensor([[0.9203, 0.3928],
        [0.0726, 0.8220]]) 



### A partir de valores aleatorios o constantes:

Podemos inicializar tensores con valores constantes. En determinadas ocasiones puede ser muy útil inicializar todos los valores al mismo. Es quizás más claro cuando inicializamos a 0. Podemos inicializar a 1 para usar una máscara en la que luego ocasionalmente cambiemos alguno de los valores a 0.
Definimos una variable `shape` para indicar las dimensiones del tensor. En este caso, 2 filas y 3 columnas.


In [ ]:
shape = (2, 3)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Tensor de aleatorios: \n {rand_tensor} \n")
print(f"Tensor de unos: \n {ones_tensor} \n")
print(f"Tensor de ceros: \n {zeros_tensor}")

Tensor de aleatorios: 
 tensor([[0.6362, 0.8958, 0.1191],
        [0.8591, 0.0888, 0.8792]]) 

Tensor de unos: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Tensor de ceros: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


## Los atributos de un tensor

Un tensor es una matriz n-dimensional, por lo tanto debe tener una forma (*shape*), y alberga siempre el mismo tipo de dato en todas sus celdas (*datatype*). Ya hemos dicho que lo que lo diferencia de un array n-dimensional de NumPy es su rol como tipo de datos para optimización de modelos.

En este caso, como sabemos, es importante el dispositivo en donde se alberga el tensor ya que existe hardware dedicado para trabajar con deep learning. Por todo esto, un tensor está ubicado en un dispositivo. Lo más frecuente es que se gestiones desde una CPU, sino en unaGPU concreta. 


In [ ]:
tensor = torch.rand(3, 4)

print(f"Shape del tensor: {tensor.shape}")
print(f"Tipo de datos ded tensor: {tensor.dtype}")
print(f"El dispositivo en el que está ubicado: {tensor.device}")

Shape del tensor: torch.Size([3, 4])
Tipo de datos ded tensor: torch.float32
El dispositivo en el que está ubicado: cpu


## Operaciones con tensores

No vamos a introducir aquí todas las operaciones que pueden hacerse con tensores porque no es el objetivo del curso. En todo caso, es bueno conocer que son un mecanismo de abstracción muy flexible y versátil. 

Prácticamente, todo lo que podemos hacer con un array NumPy, es posible también hacerlo con un tensor. Podemos trasponer, indexar, seccionar, operar matemáticamente, aplicar operaciones de álgebra lineal, muestreo aleatorio, etc. Toda la documentación oficial [es accesible aquí](https://pytorch.org/docs/stable/torch.html).

Como ya hemos mencionado, para acelerar todas estas operaciones, podemos primero mover el tensor a una GPU y después invocar la operación correspondiente. Si usamos Colab para esto, nos aseguramos primero de que nuestro notebook en Colab tiene asignada una GPU yendo a Edit \> Notebook Settings. Y seleccionando la GPU


In [ ]:
# We move our tensor to the GPU if available
if torch.cuda.is_available():
  tensor = tensor.to('cuda')
  print(f"Device tensor is stored on: {tensor.device}")

### Operaciones Numpy típicas como indexación y seccionado (slicing)

Recordemos un poco de terminología. Cuando usamos el término indexar en un array, estamos accediendo a componentes individuales concretos dentro del array. Cuando usamos el término slicing, estamos accediendo a secciones contiguas (framentos del array de más de una celda individual). 

En el ejemplo siguiente, estamos haciendo un slicing de la segunda columna de un tensor de dos dimensiones. Recordemos que con 0 nos referimos al primer elemento de la dimensión correspondiente y con 1 al segundo.

In [ ]:
tensor = torch.ones(4, 4)
tensor[:,1] = 0
print(tensor)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


### Podemos unir tensores

Podemos usar  `torch.cat` para concatenar una secuencia de tensores a lo largo de una dimensión concreta. Con la primera llamada a `cat()` lo hacemos por columnas y con la segunda por filas, al usar un tensor en 2 dimensiones.

Para una operación similar pero ligéramente diferente, ver
[torch.stack](https://pytorch.org/docs/stable/generated/torch.stack.html).

In [ ]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)
t2 = torch.cat([tensor, tensor, tensor], dim=0)
print(t2)


tensor([[1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.]])
tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


### Multiplicación de tensores

Podemos multiplicar dos matrices de las mismas dimensiones, elemento a elemento para dar una matriz de idénticas dimensiones.

In [ ]:
# This computes the element-wise product
print(f"tensor.mul(tensor) \n {tensor.mul(tensor)} \n")
# Alternative syntax:
print(f"tensor * tensor \n {tensor * tensor}")

tensor.mul(tensor) 
 tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor * tensor 
 tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


La operación que sigue es una multiplicación de matrices entre tensores.


In [ ]:
print(f"tensor.matmul(tensor.T) \n {tensor.matmul(tensor.T)} \n")
# Alternative syntax:
print(f"tensor @ tensor.T \n {tensor @ tensor.T}")

tensor.matmul(tensor.T) 
 tensor([[3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.]]) 

tensor @ tensor.T 
 tensor([[3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.]])


### Operación matricial in-place

Para realizar una operación sobre un tensor concreto, sin que genere una matriz adicional, usamos el sufijo
 `_`. Por ejemplo, las operaciones
  `x.copy_(y)`, `x.t_()` tienen como efecto el cambio directo de la variable `x`.


In [ ]:
print(tensor, "\n")
tensor.add_(5)
print(tensor)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor([[6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.]])



# Redimensionando tensores

En deep learning, es frecuente tener que redimensionar tensores. Esto lo hacemos con `reshape()` la cual aplicamos a un tensor a la que queremos cambiarle la estructura, por ejemplo para

* Aplanar una imagen y convertirla en un vector lineal, para introducirlo a un modelo de machine learning,

* la operación opuesta, para trasnformar un vector lineal en una matriz bidimensional, o tridimensional,

* Cambiar el orden de una dimensión por otra...

El ejemplo clásico es como sigue: un vector de 12 enteros generados a partir de un rango, lo convertimos en un array de 3 filas y 4 columnas, rellenando el contenedor de destino fila a fila de izquierda a derecha. Se trata de leer el array como `3 muestras, cada muestra con cuatro atributos`.

In [ ]:
import torch
x = torch.arange(12)
print(x)
y = x.reshape(3, 4)
print(y)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


Ahora uno algo más complicado, por ejemplo con 3 dimensiones. Que se leería como `2 lotes de 3 muestras cada uno, cada muestra con 4 atributos`. Fíjate además, que seguimos llenado cada lote fila a fila, de izquierda a derecha, empezando por la fila 0, y por el lote 0.

In [ ]:
x = torch.arange(24)
print(x)
y = x.reshape(2, 3, 4)
print(y)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23])
tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])


La organización de datos más común que vamos a tener en deep learning es la de 

* un conjunto de ejemplos (un lote), en donde cada ejemplo tiene un número de tokens fijo. 

* Hablamos de tokens y no números o strings, porque solemos trabajar siempre con tokens que pueden representar números, palabras, genes, cualquier cosa sensible de ser una unidad de información relevante para nuestro problema. 

* Además, cada token tiene unas dimensiones fijas, en forma de vector de números reales. Por tanto, 

Por ejemplo, 


In [ ]:
B = 2    # cells/sentences
S = 3    # tokens
D = 4    # embedding dimensions
x = torch.arange(B*S*D).reshape(B, S, D)
print(x.shape)



## Parches de una imagen para convolución

En deep learning tenemos un tipo de redes que procesan imágenes, llamadas redes convolucionales. Estas redes toman como entrada una imagen y aplican filtros (kernels) para extraer características de la imagen.

¿Cómo representamos las imágenes? Una imagen tiene una estructura naturalmente bidimensional. Pero si además es a color, lo normal es que se represente mediante RGB, con un valor para en nivel de rojo (R), otro para el verde (G) y otro para el azul (B). Por tanto, realmente necesitamos tres dimensiones, la primera con tamaño tres, y la segunda y la tercera para la altura y anchura de la imagen. Además, si tenemos lotes de imágenes, necesitaremos una dimensión 0 para indexar las imágenes.

Por simplicidad, supongamos que nuestra imagen tiene 

* 16 píxels, 

* nos viene en niveles de gris, por lo que solo necesitamos un canal de color para representarla. 

* tenemos un lote de solo una imagen.

Así, para una imagen necesitamos un tensor de $1\times 1\times 4\times 4$. 



In [ ]:
x = torch.arange(16).reshape(1, 1, 4, 4)
print(x)

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15]])


Supongamos ahora que nuestro algoritmo trabaja con parches de la figura, en lugar de con toda la figura a la vez. Necesitamos entonces trocear la figura. Supongamos que queremos hacerlo en parches de $2\times 2$. Con esto conseguimos transformar una figura en cuatro parches de ese tamaño.

In [ ]:
print(x)
x = x.reshape(1, 1, 2, 2, 2, 2)
print(x.shape)
print(x)

tensor([[[[ 0,  1,  2,  3],
          [ 4,  5,  6,  7],
          [ 8,  9, 10, 11],
          [12, 13, 14, 15]]]])
torch.Size([1, 1, 2, 2, 2, 2])
tensor([[[[[[ 0,  1],
            [ 2,  3]],

           [[ 4,  5],
            [ 6,  7]]],


          [[[ 8,  9],
            [10, 11]],

           [[12, 13],
            [14, 15]]]]]])


Es decir, 

* una dimensión para la figura dentro del lote, 

* otra para el color, 

* otra para las filas de parches dentro de la figura, 

* otra para las columnas de parches dentro de la figura, 

* otra para la fila del píxel y otra para la columna del píxel.


# Conclusiones

* Hemos visto que los tensores son como arrays n-dimensionales con una diferencia principal: sobre estos se calculan gradientes y es necesario almacenarlos junto con ellos.

* Podemos convertir Tensores desde NumPy arrays y crear NumPy arrays desde tensores cuando sea necesario

## Ejercicios sobre tensores

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>Ejercicio 1:Creación de Tensores</strong></div>

* Crea un tensor unidimensional con los valores [1, 2, 3, 4, 5].

* Crea un tensor bidimensional de tamaño (3, 3) lleno de números aleatorios.

* Crea un tensor de ceros con tamaño (4, 4).

* Imprime el tamaño, el tipo de datos y el dispositivo (CPU/GPU) de cada tensor.



<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>Ejercicio 2:Operaciones Básicas con Tensores</strong></div>

* Crea dos tensores de tamaño (2, 2) con valores a tu elección.

* Suma los dos tensores.

* Multiplica los dos tensores elemento por elemento.

* Calcula el producto matricial de los dos tensores.







<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>Ejercicio 3: Manipulación de Tensores</strong></div>

* Crea un tensor de tamaño (4, 4) inicializado con números aleatorios.

* Cambia la forma del tensor a (2, 8).

* Concatenar dos tensores de tamaño (2, 2) a lo largo de la dimensión 0.

* Obtén la suma de todos los elementos de un tensor.


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>Ejercicio 4: Extracción de Valores de un Tensor</strong></div>

* Crea un tensor escalar (un solo número).

* Extrae el valor como un escalar de Python usando .item().

* Crea un tensor de múltiples dimensiones y conviértelo en un arreglo NumPy.

* Extrae un valor específico del tensor multidimensional.

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>Ejercicio 5: Indexación y Slicing en Tensores</strong></div>

* Crea un tensor de tamaño (4, 4) con números enteros secuenciales del 1 al 16.

* Extrae el elemento en la fila 2, columna 3.

* Extrae la segunda fila completa.

* Extrae las dos primeras columnas de todas las filas.

* Cambia el valor en la posición (3, 3) a 99.

* Extrae un subtensor que contenga las filas 1 y 2 y las columnas 2 y 3.

* Obtén un tensor que contenga la penúltima columna

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>Ejercicio 6: El ejemplo de la convolución</strong></div>

Explicar qué es lo que realmente no ha ido bien en el ejemplo de la convolución y los parches de $2\times 2$. Dar una pista de cómo se podría arreglar usando [permute()](https://docs.pytorch.org/docs/2.14/generated/torch.permute.html).




In [ ]:
x = torch.arange(16).reshape(1, 1, 4, 4)
print(x)
x = x.reshape(2, 2, 2, 2)
print(x)    
patches = (
    x.reshape(2, 2, 2, 2)
     .permute(0, 2, 1, 3)
     .reshape(4, 4)
)
print(patches)

tensor([[[[ 0,  1,  2,  3],
          [ 4,  5,  6,  7],
          [ 8,  9, 10, 11],
          [12, 13, 14, 15]]]])
tensor([[[[ 0,  1],
          [ 2,  3]],

         [[ 4,  5],
          [ 6,  7]]],


        [[[ 8,  9],
          [10, 11]],

         [[12, 13],
          [14, 15]]]])
tensor([[ 0,  1,  4,  5],
        [ 2,  3,  6,  7],
        [ 8,  9, 12, 13],
        [10, 11, 14, 15]])
